## DCF Valuation Modeling

In [8]:
import pandas as pd
import numpy as np

# =========================================================
# BASE ASSUMPTIONS
# =========================================================
ASSUMPTIONS = {
    "first_year_forecast": 2023,
    "historical_years": [2020, 2021, 2022],
    "forecast_years": [2023, 2024, 2025, 2026, 2027],
    "terminal_label": "Term",

    "base_ebitda_2022": 18379,
    "growth_rates": [0.075, 0.070, 0.060, 0.050, 0.040],
    "terminal_growth_rate": 0.020,

    "tax_depreciation": [3700, 3700, 3700, 3700, 3700],
    "terminal_tax_depreciation": 3700,

    "tax_rates": [0.17, 0.17, 0.17, 0.17, 0.17],
    "terminal_tax_rate": 0.17,

    "cash_from_working_capital": [-100, -100, -100, -100, -100],
    "terminal_cash_from_working_capital": -100,

    "wacc": 0.135,
    "net_debt": 18642,
    "shares_outstanding": 34200,
    "current_price": 2.71,
}

# Optional historical values for display only
HISTORICAL = {
    2020: {"EBITDA": 16247, "Tax Depreciation": 2960, "Current Tax": 2458},
    2021: {"EBITDA": 17128, "Tax Depreciation": 3196, "Current Tax": 2270.4},
    2022: {"EBITDA": 18379, "Tax Depreciation": 3452, "Current Tax": 2358},
}


# =========================================================
# HELPER FUNCTIONS
# =========================================================
def fmt_num(x, decimals=0):
    if pd.isna(x):
        return ""
    return f"{x:,.{decimals}f}"

def fmt_pct(x, decimals=1):
    if pd.isna(x):
        return ""
    return f"{x*100:.{decimals}f}%"

def fmt_neg_paren(x, decimals=0):
    if pd.isna(x):
        return ""
    if x < 0:
        return f"({abs(x):,.{decimals}f})"
    return f"{x:,.{decimals}f}"


# =========================================================
# CORE DCF ENGINE
# =========================================================
def build_ufcf_schedule(inputs):
    years = inputs["forecast_years"]
    growth_rates = inputs["growth_rates"]

    ebitda = []
    current = inputs["base_ebitda_2022"]
    for g in growth_rates:
        current = current * (1 + g)
        ebitda.append(current)

    tax_dep = inputs["tax_depreciation"]
    operating_profit = [e - d for e, d in zip(ebitda, tax_dep)]
    tax_rates = inputs["tax_rates"]
    current_tax = [op * t for op, t in zip(operating_profit, tax_rates)]
    capex = [-d for d in tax_dep]
    wcf = inputs["cash_from_working_capital"]

    ufcf = [
        e - tax + cap + wc
        for e, tax, cap, wc in zip(ebitda, current_tax, capex, wcf)
    ]

    # Terminal year
    terminal_ebitda = ebitda[-1] * (1 + inputs["terminal_growth_rate"])
    terminal_tax_dep = inputs["terminal_tax_depreciation"]
    terminal_operating_profit = terminal_ebitda - terminal_tax_dep
    terminal_tax = terminal_operating_profit * inputs["terminal_tax_rate"]
    terminal_capex = -terminal_tax_dep
    terminal_wcf = inputs["terminal_cash_from_working_capital"]
    terminal_ufcf = terminal_ebitda - terminal_tax + terminal_capex + terminal_wcf

    ufcf_df = pd.DataFrame({
        "Metric": [
            "EBITDA Growth",
            "EBITDA",
            "Tax Depreciation",
            "Operating Profit",
            "Tax Rate",
            "Current Tax",
            "Capital Expenditure",
            "Cash from Working Capital",
            "Unlevered Free Cash Flow",
        ]
    })

    for i, y in enumerate(years):
        ufcf_df[y] = [
            growth_rates[i],
            ebitda[i],
            tax_dep[i],
            operating_profit[i],
            tax_rates[i],
            current_tax[i],
            -tax_dep[i],
            wcf[i],
            ufcf[i],
        ]

    ufcf_df["Term"] = [
        inputs["terminal_growth_rate"],
        terminal_ebitda,
        terminal_tax_dep,
        terminal_operating_profit,
        inputs["terminal_tax_rate"],
        terminal_tax,
        terminal_capex,
        terminal_wcf,
        terminal_ufcf,
    ]

    return {
        "years": years,
        "ebitda": ebitda,
        "operating_profit": operating_profit,
        "current_tax": current_tax,
        "capex": capex,
        "working_capital": wcf,
        "ufcf": ufcf,
        "terminal_ufcf": terminal_ufcf,
        "ufcf_schedule": ufcf_df
    }


def dcf_valuation(inputs, ufcf_data):
    wacc = inputs["wacc"]
    g = inputs["terminal_growth_rate"]
    ufcf = ufcf_data["ufcf"]
    terminal_ufcf = ufcf_data["terminal_ufcf"]

    # Match Excel logic:
    # PV of discrete = NPV(WACC, K48:O48)
    # PV of terminal = NPV(WACC, K49:O49), where only last year has terminal value
    pv_discrete = sum(cf / ((1 + wacc) ** i) for i, cf in enumerate(ufcf, start=1))

    terminal_value = terminal_ufcf / (wacc - g)
    pv_terminal = terminal_value / ((1 + wacc) ** len(ufcf))

    enterprise_value = pv_discrete + pv_terminal
    equity_value = enterprise_value - inputs["net_debt"]
    equity_value_per_share = equity_value / inputs["shares_outstanding"]
    premium_discount = equity_value_per_share / inputs["current_price"] - 1

    cash_flow_profiles = pd.DataFrame({
        "Metric": ["Discrete Forecast", "Terminal Value", "Total Cash Flow"]
    })

    for i, y in enumerate(inputs["forecast_years"]):
        cash_flow_profiles[y] = [
            ufcf[i],
            0 if i < len(ufcf) - 1 else terminal_value,
            ufcf[i] if i < len(ufcf) - 1 else ufcf[i] + terminal_value
        ]

    dcf_summary = {
        "first_year_forecast": inputs["first_year_forecast"],
        "terminal_growth_rate": inputs["terminal_growth_rate"],
        "wacc": inputs["wacc"],
        "pv_discrete": pv_discrete,
        "pv_terminal": pv_terminal,
        "enterprise_value": enterprise_value,
        "net_debt": inputs["net_debt"],
        "equity_value": equity_value,
        "shares_outstanding": inputs["shares_outstanding"],
        "equity_value_per_share": equity_value_per_share,
        "current_price": inputs["current_price"],
        "premium_discount": premium_discount,
        "terminal_value": terminal_value,
        "cash_flow_profiles": cash_flow_profiles,
    }

    return dcf_summary


def build_sensitivity_tables(inputs, base_ufcf):
    base_wacc = inputs["wacc"]
    base_g = inputs["terminal_growth_rate"]

    wacc_range = [base_wacc - 0.02, base_wacc - 0.01, base_wacc, base_wacc + 0.01, base_wacc + 0.02]
    g_range = [base_g - 0.01, base_g - 0.005, base_g, base_g + 0.005, base_g + 0.01]

    enterprise_table = pd.DataFrame(index=wacc_range, columns=g_range, dtype=float)
    equity_table = pd.DataFrame(index=wacc_range, columns=g_range, dtype=float)
    per_share_table = pd.DataFrame(index=wacc_range, columns=g_range, dtype=float)
    premium_table = pd.DataFrame(index=wacc_range, columns=g_range, dtype=float)

    discrete_cf = base_ufcf["ufcf"]
    terminal_base = base_ufcf["terminal_ufcf"]

    for w in wacc_range:
        pv_discrete = sum(cf / ((1 + w) ** i) for i, cf in enumerate(discrete_cf, start=1))

        for g in g_range:
            # Rebuild terminal UFCF using altered terminal growth
            terminal_ufcf = discrete_cf[-1] * (1 + g)
            terminal_value = terminal_ufcf / (w - g)
            pv_terminal = terminal_value / ((1 + w) ** len(discrete_cf))

            ev = pv_discrete + pv_terminal
            eq = ev - inputs["net_debt"]
            ps = eq / inputs["shares_outstanding"]
            prem = ps / inputs["current_price"] - 1

            enterprise_table.loc[w, g] = ev
            equity_table.loc[w, g] = eq
            per_share_table.loc[w, g] = ps
            premium_table.loc[w, g] = prem

    return {
        "wacc_range": wacc_range,
        "g_range": g_range,
        "enterprise_value": enterprise_table,
        "equity_value": equity_table,
        "equity_value_per_share": per_share_table,
        "premium_discount": premium_table
    }


# =========================================================
# REPORT BUILDERS
# =========================================================
def print_ufcf_schedule(inputs, ufcf_data):
    print("\n" + "=" * 80)
    print("1. UNLEVERED FREE CASH FLOW SCHEDULE")
    print("=" * 80)

    df = ufcf_data["ufcf_schedule"].copy()

    formatted = df.copy()
    formatted.iloc[0, 1:] = formatted.iloc[0, 1:].apply(lambda x: fmt_pct(x, 1))
    formatted.iloc[1, 1:] = formatted.iloc[1, 1:].apply(lambda x: fmt_num(x, 0))
    formatted.iloc[2, 1:] = formatted.iloc[2, 1:].apply(lambda x: fmt_num(x, 0))
    formatted.iloc[3, 1:] = formatted.iloc[3, 1:].apply(lambda x: fmt_num(x, 0))
    formatted.iloc[4, 1:] = formatted.iloc[4, 1:].apply(lambda x: fmt_pct(x, 0))
    formatted.iloc[5, 1:] = formatted.iloc[5, 1:].apply(lambda x: fmt_num(x, 0))
    formatted.iloc[6, 1:] = formatted.iloc[6, 1:].apply(lambda x: fmt_neg_paren(x, 0))
    formatted.iloc[7, 1:] = formatted.iloc[7, 1:].apply(lambda x: fmt_neg_paren(x, 0))
    formatted.iloc[8, 1:] = formatted.iloc[8, 1:].apply(lambda x: fmt_num(x, 0))

    print(formatted.to_string(index=False))


def print_dcf_schedule(inputs, dcf_data):
    print("\n" + "=" * 80)
    print("2. DISCOUNTED CASH FLOW SCHEDULE")
    print("=" * 80)

    print("\nASSUMPTIONS")
    print(f"First Year of Forecast : {dcf_data['first_year_forecast']}")
    print(f"Terminal Growth Rate   : {fmt_pct(dcf_data['terminal_growth_rate'], 1)}")
    print(f"WACC                   : {fmt_pct(dcf_data['wacc'], 1)}")

    print("\nENTERPRISE VALUE")
    print(f"PV of Discrete         : {fmt_num(dcf_data['pv_discrete'], 0)}")
    print(f"PV of Terminal         : {fmt_num(dcf_data['pv_terminal'], 0)}")
    print(f"Enterprise Value       : {fmt_num(dcf_data['enterprise_value'], 0)}")

    print("\nEQUITY VALUE")
    print(f"Enterprise Value       : {fmt_num(dcf_data['enterprise_value'], 0)}")
    print(f"Less: Net Debt         : ({fmt_num(dcf_data['net_debt'], 0)})")
    print(f"Equity Value           : {fmt_num(dcf_data['equity_value'], 0)}")

    print("\nEQUITY VALUE PER SHARE")
    print(f"Equity Value           : {fmt_num(dcf_data['equity_value'], 0)}")
    print(f"Shares Outstanding     : {fmt_num(dcf_data['shares_outstanding'], 0)}")
    print(f"Equity Value / Share   : {dcf_data['equity_value_per_share']:.2f}")

    print("\nPREMIUM (DISCOUNT)")
    print(f"Equity Value / Share   : {dcf_data['equity_value_per_share']:.2f}")
    print(f"Current Price          : {dcf_data['current_price']:.2f}")
    print(f"Premium (Discount)     : {fmt_pct(dcf_data['premium_discount'], 1)}")

    print("\nCASH FLOW PROFILES")
    cf = dcf_data["cash_flow_profiles"].copy()
    for col in cf.columns[1:]:
        cf[col] = cf[col].apply(lambda x: fmt_num(x, 0))
    print(cf.to_string(index=False))


def print_sensitivity_tables(sens):
    print("\n" + "=" * 80)
    print("3. SENSITIVITY ANALYSIS")
    print("=" * 80)

    def format_table(df, value_type="num"):
        out = df.copy()
        out.index = [fmt_pct(i, 1) for i in out.index]
        out.columns = [fmt_pct(c, 1) for c in out.columns]

        for r in out.index:
            for c in out.columns:
                val = out.loc[r, c]
                if value_type == "num":
                    out.loc[r, c] = fmt_num(val, 0)
                elif value_type == "per_share":
                    out.loc[r, c] = f"{val:.2f}"
                elif value_type == "pct":
                    out.loc[r, c] = fmt_pct(val, 1)
        return out

    print("\nENTERPRISE VALUE")
    print(format_table(sens["enterprise_value"], "num").to_string())

    print("\nEQUITY VALUE")
    print(format_table(sens["equity_value"], "num").to_string())

    print("\nEQUITY VALUE PER SHARE")
    print(format_table(sens["equity_value_per_share"], "per_share").to_string())

    print("\nPREMIUM (DISCOUNT) TO CURRENT PRICE")
    print(format_table(sens["premium_discount"], "pct").to_string())


# =========================================================
# RUN FULL MODEL
# =========================================================
ufcf_data = build_ufcf_schedule(ASSUMPTIONS)
dcf_data = dcf_valuation(ASSUMPTIONS, ufcf_data)
sens_data = build_sensitivity_tables(ASSUMPTIONS, ufcf_data)

print_ufcf_schedule(ASSUMPTIONS, ufcf_data)
print_dcf_schedule(ASSUMPTIONS, dcf_data)
print_sensitivity_tables(sens_data)


1. UNLEVERED FREE CASH FLOW SCHEDULE
                   Metric    2023    2024    2025    2026    2027    Term
            EBITDA Growth    7.5%    7.0%    6.0%    5.0%    4.0%    2.0%
                   EBITDA  19,757  21,140  22,409  23,529  24,470  24,960
         Tax Depreciation   3,700   3,700   3,700   3,700   3,700   3,700
         Operating Profit  16,057  17,440  18,709  19,829  20,770  21,260
                 Tax Rate     17%     17%     17%     17%     17%     17%
              Current Tax   2,730   2,965   3,181   3,371   3,531   3,614
      Capital Expenditure (3,700) (3,700) (3,700) (3,700) (3,700) (3,700)
Cash from Working Capital   (100)   (100)   (100)   (100)   (100)   (100)
 Unlevered Free Cash Flow  13,228  14,376  15,428  16,358  17,140  17,546

2. DISCOUNTED CASH FLOW SCHEDULE

ASSUMPTIONS
First Year of Forecast : 2023
Terminal Growth Rate   : 2.0%
WACC                   : 13.5%

ENTERPRISE VALUE
PV of Discrete         : 52,322
PV of Terminal         : 81,002
En

/var/folders/c4/xntd0yb16dz1d4dv_zrhdtvc0000gn/T/ipykernel_61590/682963166.py:252: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7.5%' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  formatted.iloc[0, 1:] = formatted.iloc[0, 1:].apply(lambda x: fmt_pct(x, 1))
/var/folders/c4/xntd0yb16dz1d4dv_zrhdtvc0000gn/T/ipykernel_61590/682963166.py:252: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7.0%' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  formatted.iloc[0, 1:] = formatted.iloc[0, 1:].apply(lambda x: fmt_pct(x, 1))
/var/folders/c4/xntd0yb16dz1d4dv_zrhdtvc0000gn/T/ipykernel_61590/682963166.py:252: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '6.0%' has dtype 